# DTDIF — Final Complete Colab + Google Drive Notebook

**Digital Twin Decision Intelligence Framework**

This notebook is designed for **Google Colab** and saves all outputs to **Google Drive**.

It creates:

- `/content/drive/MyDrive/Outputs/DTDIF_Final/figures`
- `/content/drive/MyDrive/Outputs/DTDIF_Final/tables`
- `/content/drive/MyDrive/Outputs/DTDIF_Final/models`
- `/content/drive/MyDrive/Outputs/DTDIF_Final/outputs`
- `/content/drive/MyDrive/Outputs/DTDIF_Final/outputs/outputs_summary.txt`

It also generates publication-oriented figures and tables for the article.

In [ ]:
# ============================================================
# Cell 1 — Environment setup
# ============================================================

import os
import json
import time
import math
import random
import pickle
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    make_scorer,
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

try:
    import networkx as nx
    HAS_NETWORKX = True
except Exception:
    HAS_NETWORKX = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Environment ready.")
print("Timestamp:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

In [ ]:
# ============================================================
# Cell 2 — Google Drive mount and output directories
# ============================================================

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    BASE_DIR = Path("/content/drive/MyDrive/Outputs/DTDIF_Final")
else:
    BASE_DIR = Path.cwd() / "Outputs" / "DTDIF_Final"

FIG_DIR = BASE_DIR / "figures"
TABLE_DIR = BASE_DIR / "tables"
MODEL_DIR = BASE_DIR / "models"
OUTPUT_DIR = BASE_DIR / "outputs"
DATA_DIR = BASE_DIR / "data"
LOG_DIR = BASE_DIR / "logs"

for p in [BASE_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR, OUTPUT_DIR, DATA_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = OUTPUT_DIR / "outputs_summary.txt"

def savefig(name):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved figure:", path)
    return path

def save_table(df, name):
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    print("Saved table:", path)
    return path

def save_json(obj, name):
    path = OUTPUT_DIR / name
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)
    print("Saved JSON:", path)
    return path

print("IN_COLAB:", IN_COLAB)
print("BASE_DIR:", BASE_DIR)
print("FIG_DIR:", FIG_DIR)
print("TABLE_DIR:", TABLE_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# ============================================================
# Cell 3 — Configuration
# ============================================================

CONFIG = {
    "framework_name": "Digital Twin Decision Intelligence Framework",
    "framework_abbreviation": "DTDIF",
    "experiment_name": "dtdif_final_complete_colab_drive",
    "seed": SEED,
    "test_size": 0.25,
    "n_splits": 5,
    "n_synthetic_cyber": 3500,
    "n_synthetic_video": 2800,

    "cyber_generator": {
        "n_classes": 4,
        "n_features": 20,
        "n_informative": 12,
        "n_redundant": 5,
        "class_sep": 1.45,
        "flip_y": 0.025,
        "weights": [0.55, 0.22, 0.15, 0.08],
    },

    "video_generator": {
        "n_classes": 2,
        "n_features": 16,
        "n_informative": 5,
        "n_redundant": 5,
        "n_clusters_per_class": 3,
        "class_sep": 0.42,
        "flip_y": 0.18,
        "weights": [0.72, 0.28],
    },

    "free_energy_weights": {
        "uncertainty": 0.30,
        "severity": 0.35,
        "context": 0.20,
        "model_risk": 0.15,
        "confidence_penalty": 0.10,
    },

    "decision_thresholds": {
        "monitor": 0.35,
        "investigate": 0.55,
        "contain": 0.72,
        "escalate": 0.86,
    },

    "twin_dynamics": {
        "passive_recovery": 0.020,
        "risk_drain": 0.060,
        "mitigation_gain": 0.075,
        "workload_penalty": 0.035,
        "workload_decay": 0.90,
        "risk_decay": 0.96,
        "risk_volume_scaling": True,
    },

    "max_permutation_features": 12,
}

save_json(CONFIG, "config.json")
CONFIG

## Dataset strategy

The notebook can run immediately with synthetic data.  
It also includes hooks to load real CSV datasets from Google Drive if available.

Recommended real datasets for later replacement:

- UNSW-NB15
- CICIDS2017 / CICIDS2018
- TON-IoT
- Bot-IoT
- real surveillance event logs or extracted scene descriptors

In [ ]:
# ============================================================
# Cell 4 — Dataset loading hooks
# ============================================================

def discover_csv_candidates():
    roots = [
        DATA_DIR,
        Path("/content/drive/MyDrive/Datasets") if IN_COLAB else Path.cwd(),
        Path("/content/drive/MyDrive") if IN_COLAB else Path.cwd(),
        Path.cwd(),
    ]
    files = []
    for root in roots:
        if root.exists():
            try:
                files.extend(list(root.rglob("*.csv"))[:500])
            except Exception:
                pass
    return sorted(set(files))

def infer_target_column(df):
    candidates = [
        "label", "target", "class", "attack", "anomaly", "is_attack",
        "y", "intrusion", "event_label", "risk_label", "threat",
        "severity", "incident_class", "category"
    ]
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in lower_map:
            return lower_map[c]
    for c in reversed(df.columns):
        if 2 <= df[c].nunique(dropna=True) <= 10:
            return c
    return None

def load_candidate_dataset(keywords):
    files = discover_csv_candidates()
    scored = []
    for f in files:
        score = sum(k in f.name.lower() for k in keywords)
        if score > 0:
            scored.append((score, f))
    scored = sorted(scored, reverse=True)

    for _, f in scored:
        try:
            df = pd.read_csv(f)
            if len(df) > 200 and infer_target_column(df) is not None:
                print("Loaded real/candidate dataset:", f)
                return df, f
        except Exception:
            continue

    return None, None

print("CSV candidates found:", len(discover_csv_candidates()))

In [ ]:
# ============================================================
# Cell 5 — Synthetic dataset generation
# ============================================================

def generate_synthetic_cyber_dataset(n_samples=3500, seed=42):
    cfg = CONFIG["cyber_generator"]
    X, y = make_classification(
        n_samples=n_samples,
        n_features=cfg["n_features"],
        n_informative=cfg["n_informative"],
        n_redundant=cfg["n_redundant"],
        n_repeated=0,
        n_classes=cfg["n_classes"],
        n_clusters_per_class=1,
        weights=cfg["weights"],
        class_sep=cfg["class_sep"],
        flip_y=cfg["flip_y"],
        random_state=seed,
    )

    cols = [
        "packet_rate", "byte_rate", "flow_duration", "src_entropy", "dst_entropy",
        "failed_logins", "connection_burst", "port_diversity", "protocol_entropy",
        "payload_irregularity", "dns_query_rate", "http_error_rate", "syn_ratio",
        "ack_ratio", "inbound_outbound_ratio", "session_reuse", "geo_deviation",
        "device_trust_score", "lateral_movement_score", "privilege_escalation_score"
    ]

    rng = np.random.default_rng(seed)
    df = pd.DataFrame(X, columns=cols)
    df["hour"] = rng.integers(0, 24, size=n_samples)
    df["asset_criticality"] = rng.choice([1, 2, 3, 4, 5], size=n_samples, p=[0.20, 0.25, 0.25, 0.20, 0.10])
    df["label"] = y
    return df

def generate_synthetic_video_dataset(n_samples=2800, seed=43):
    cfg = CONFIG["video_generator"]
    X, y = make_classification(
        n_samples=n_samples,
        n_features=cfg["n_features"],
        n_informative=cfg["n_informative"],
        n_redundant=cfg["n_redundant"],
        n_repeated=0,
        n_classes=cfg["n_classes"],
        n_clusters_per_class=cfg["n_clusters_per_class"],
        weights=cfg["weights"],
        class_sep=cfg["class_sep"],
        flip_y=cfg["flip_y"],
        random_state=seed,
    )

    cols = [
        "motion_density", "object_count", "crowd_irregularity", "trajectory_deviation",
        "loitering_score", "zone_crossing_rate", "occlusion_ratio", "scene_change_rate",
        "camera_blur", "illumination_shift", "abandoned_object_score", "speed_variance",
        "pose_anomaly", "camera_trust_score", "background_instability", "object_persistence"
    ]

    rng = np.random.default_rng(seed)
    df = pd.DataFrame(X, columns=cols)
    df["hour"] = rng.integers(0, 24, size=n_samples)
    df["zone_criticality"] = rng.choice([1, 2, 3, 4, 5], size=n_samples, p=[0.18, 0.28, 0.26, 0.18, 0.10])
    df["label"] = y
    return df

cyber_df, cyber_path = load_candidate_dataset(["cyber", "network", "ids", "intrusion", "attack", "unsw", "cicids"])
video_df, video_path = load_candidate_dataset(["video", "surveillance", "camera", "event"])

if cyber_df is None:
    cyber_df = generate_synthetic_cyber_dataset(CONFIG["n_synthetic_cyber"], SEED)
    cyber_path = DATA_DIR / "synthetic_cyber_final.csv"
    cyber_df.to_csv(cyber_path, index=False)
    print("Generated synthetic cyber dataset:", cyber_path)

if video_df is None:
    video_df = generate_synthetic_video_dataset(CONFIG["n_synthetic_video"], SEED + 1)
    video_path = DATA_DIR / "synthetic_video_final.csv"
    video_df.to_csv(video_path, index=False)
    print("Generated synthetic video dataset:", video_path)

dataset_overview = pd.DataFrame([
    {"dataset": "cyber", "path": str(cyber_path), "samples": len(cyber_df), "features_plus_target": cyber_df.shape[1]},
    {"dataset": "video", "path": str(video_path), "samples": len(video_df), "features_plus_target": video_df.shape[1]},
])
save_table(dataset_overview, "table_dataset_overview.csv")

display(dataset_overview)
display(cyber_df.head())
display(video_df.head())

In [ ]:
# ============================================================
# Cell 6 — Data preparation
# ============================================================

def prepare_xy(df):
    target_col = infer_target_column(df)
    if target_col is None:
        raise ValueError("No target column found.")

    y_raw = df[target_col]
    X = df.drop(columns=[target_col]).copy()

    y = pd.factorize(y_raw)[0]

    for c in X.columns:
        if X[c].dtype == object:
            X[c] = pd.factorize(X[c].astype(str))[0]

    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True)).fillna(0)

    return X, pd.Series(y, name="label")

X_cyber, y_cyber = prepare_xy(cyber_df)
X_video, y_video = prepare_xy(video_df)

target_distribution = pd.concat([
    y_cyber.value_counts(normalize=True).sort_index().rename("ratio").reset_index().assign(dataset="cyber").rename(columns={"index": "class"}),
    y_video.value_counts(normalize=True).sort_index().rename("ratio").reset_index().assign(dataset="video").rename(columns={"index": "class"}),
], ignore_index=True)

save_table(target_distribution, "table_target_distribution.csv")
display(target_distribution)

In [ ]:
# ============================================================
# Cell 7 — Model zoo and metrics
# ============================================================

def get_model_zoo():
    return {
        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1500, class_weight="balanced", random_state=SEED))
        ]),
        "RandomForest": RandomForestClassifier(
            n_estimators=260,
            min_samples_leaf=3,
            class_weight="balanced_subsample",
            random_state=SEED,
            n_jobs=-1,
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=320,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=220,
            learning_rate=0.04,
            max_depth=3,
            random_state=SEED,
        ),
    }

def compute_metrics(y_true, proba):
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)
    n_classes = len(np.unique(y_true))

    if proba.shape[1] == 2:
        p1 = proba[:, 1]
        pred = (p1 >= 0.5).astype(int)
        return {
            "accuracy": accuracy_score(y_true, pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "roc_auc": roc_auc_score(y_true, p1),
            "pr_auc": average_precision_score(y_true, p1),
            "brier": brier_score_loss(y_true, p1),
        }

    pred = np.argmax(proba, axis=1)

    try:
        roc_auc = roc_auc_score(y_true, proba, multi_class="ovr", average="weighted")
    except Exception:
        roc_auc = np.nan

    try:
        y_onehot = pd.get_dummies(y_true).reindex(columns=range(proba.shape[1]), fill_value=0).values
        pr_auc = average_precision_score(y_onehot, proba, average="weighted")
    except Exception:
        pr_auc = np.nan

    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, average="weighted", zero_division=0),
        "recall": recall_score(y_true, pred, average="weighted", zero_division=0),
        "f1": f1_score(y_true, pred, average="weighted", zero_division=0),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "brier": np.nan,
    }

In [ ]:
# ============================================================
# Cell 8 — Train models and select best model
# ============================================================

def train_and_select(X, y, dataset_name):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=CONFIG["test_size"],
        stratify=y,
        random_state=SEED
    )

    rows = []
    fitted = {}

    print("\nTraining models for:", dataset_name)

    for model_name, model in get_model_zoo().items():
        t0 = time.time()

        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)
        m = compute_metrics(y_test, proba)

        m.update({
            "dataset": dataset_name,
            "model": model_name,
            "n_classes": len(np.unique(y)),
            "time_sec": time.time() - t0,
        })

        rows.append(m)
        fitted[model_name] = model

        print(f"{dataset_name} | {model_name}: F1={m['f1']:.3f}, ROC-AUC={m['roc_auc']:.3f}")

    results_df = pd.DataFrame(rows).sort_values(["f1", "roc_auc"], ascending=False).reset_index(drop=True)
    best_name = results_df.loc[0, "model"]
    best_model = fitted[best_name]
    best_proba = best_model.predict_proba(X_test)

    return {
        "dataset": dataset_name,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "results_df": results_df,
        "best_name": best_name,
        "best_model": best_model,
        "best_proba": best_proba,
        "n_classes": len(np.unique(y)),
    }

cyber_run = train_and_select(X_cyber, y_cyber, "cyber")
video_run = train_and_select(X_video, y_video, "video")

metrics_df = pd.concat([cyber_run["results_df"], video_run["results_df"]], ignore_index=True)
save_table(metrics_df, "table_model_performance.csv")

best_model_summary = metrics_df.sort_values(["dataset", "f1", "roc_auc"], ascending=[True, False, False]).groupby("dataset").head(1)
save_table(best_model_summary, "table_best_model_summary.csv")

display(metrics_df)
display(best_model_summary)

In [ ]:
# ============================================================
# Cell 9 — Cross-validation stability
# ============================================================

def cv_stability(X, y, model, dataset_name):
    n_classes = len(np.unique(y))
    if n_classes == 2:
        scoring = {
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "f1": "f1",
            "roc_auc": "roc_auc",
        }
    else:
        scoring = {
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "f1_weighted": make_scorer(f1_score, average="weighted", zero_division=0),
        }

    cv = StratifiedKFold(n_splits=CONFIG["n_splits"], shuffle=True, random_state=SEED)
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)

    rows = []
    for metric in scoring:
        vals = scores[f"test_{metric}"]
        rows.append({
            "dataset": dataset_name,
            "metric": metric,
            "mean": vals.mean(),
            "std": vals.std(),
            "min": vals.min(),
            "max": vals.max(),
        })

    return pd.DataFrame(rows)

cv_df = pd.concat([
    cv_stability(X_cyber, y_cyber, cyber_run["best_model"], "cyber"),
    cv_stability(X_video, y_video, video_run["best_model"], "video"),
], ignore_index=True)

save_table(cv_df, "table_cv_stability.csv")
display(cv_df)

## Variational free-energy decision layer

The DTDIF decision layer is expressed as:

\[
\mathcal{F}(x)
=
\lambda_U U(x)
+
\lambda_S S(x)
+
\lambda_C C(x)
+
\lambda_R R(x)
+
\lambda_Q Q(x),
\]

where:

- \(U(x)\): predictive uncertainty;
- \(S(x)\): severity potential;
- \(C(x)\): contextual criticality;
- \(R(x)\): model-derived risk;
- \(Q(x)=1-\text{confidence}(x)\): confidence penalty.

The adaptive free energy is then mapped to operational actions:
observe, monitor, investigate, contain, and escalate.

In [ ]:
# ============================================================
# Cell 10 — Variational free-energy decision layer
# ============================================================

def normalize01(x):
    x = np.asarray(x, dtype=float)
    mn, mx = np.nanmin(x), np.nanmax(x)
    if mx - mn < 1e-12:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)

def entropy_from_proba(proba):
    proba = np.asarray(proba, dtype=float)
    proba = np.clip(proba, 1e-8, 1.0)
    ent = -np.sum(proba * np.log(proba), axis=1)
    return ent / np.log(proba.shape[1])

def severity_potential(X):
    scaled = StandardScaler().fit_transform(X)
    magnitude = np.abs(scaled).mean(axis=1)
    tailness = np.percentile(np.abs(scaled), 90, axis=1)
    return normalize01(0.65 * magnitude + 0.35 * tailness)

def model_risk_from_proba(proba):
    if proba.shape[1] == 2:
        return proba[:, 1]
    return 1.0 - proba[:, 0]

def compute_decision_frame(run, context_candidates):
    X_test = run["X_test"].copy()
    proba = np.asarray(run["best_proba"])

    U = entropy_from_proba(proba)
    R = model_risk_from_proba(proba)
    confidence = np.max(proba, axis=1)
    Q = 1.0 - confidence
    S = severity_potential(X_test)

    context_col = None
    for c in context_candidates:
        if c in X_test.columns:
            context_col = c
            break

    if context_col:
        C = normalize01(X_test[context_col])
    else:
        C = np.zeros(len(X_test))

    w = CONFIG["free_energy_weights"]

    F = (
        w["uncertainty"] * U +
        w["severity"] * S +
        w["context"] * C +
        w["model_risk"] * R +
        w["confidence_penalty"] * Q
    )

    adaptive_F = normalize01(0.80 * F + 0.15 * R + 0.05 * U)

    thresholds = CONFIG["decision_thresholds"]

    actions = []
    for x in adaptive_F:
        if x >= thresholds["escalate"]:
            actions.append("escalate")
        elif x >= thresholds["contain"]:
            actions.append("contain")
        elif x >= thresholds["investigate"]:
            actions.append("investigate")
        elif x >= thresholds["monitor"]:
            actions.append("monitor")
        else:
            actions.append("observe")

    df = pd.DataFrame({
        "dataset": run["dataset"],
        "y_true": np.asarray(run["y_test"]),
        "uncertainty": U,
        "severity_potential": S,
        "context_criticality": C,
        "model_risk": R,
        "model_confidence": confidence,
        "confidence_penalty": Q,
        "free_energy": F,
        "adaptive_free_energy": adaptive_F,
        "action": actions,
    })

    return df

cyber_decision_df = compute_decision_frame(cyber_run, ["asset_criticality", "criticality"])
video_decision_df = compute_decision_frame(video_run, ["zone_criticality", "criticality"])

decision_df = pd.concat([cyber_decision_df, video_decision_df], ignore_index=True)
save_table(decision_df, "table_decision_layer_outputs.csv")

action_counts = decision_df.groupby(["dataset", "action"]).size().reset_index(name="count")
save_table(action_counts, "table_action_counts.csv")

decision_summary = decision_df.groupby("dataset").agg(
    mean_free_energy=("free_energy", "mean"),
    mean_adaptive_free_energy=("adaptive_free_energy", "mean"),
    mean_uncertainty=("uncertainty", "mean"),
    mean_model_risk=("model_risk", "mean"),
    mean_model_confidence=("model_confidence", "mean"),
    mean_severity_potential=("severity_potential", "mean"),
    high_risk_rate=("adaptive_free_energy", lambda x: float(np.mean(x >= CONFIG["decision_thresholds"]["contain"]))),
).reset_index()

save_table(decision_summary, "table_decision_summary.csv")

display(decision_df.head())
display(action_counts)
display(decision_summary)

In [ ]:
# ============================================================
# Cell 11 — Digital twin resilience simulation
# ============================================================

def simulate_digital_twin(decision_df, dataset_name):
    df = decision_df[decision_df["dataset"] == dataset_name].copy().reset_index(drop=True)
    df = df.sort_values("adaptive_free_energy", ascending=False).reset_index(drop=True)

    dyn = CONFIG["twin_dynamics"]

    mitigation = {
        "observe": 0.004,
        "monitor": 0.020,
        "investigate": 0.050,
        "contain": 0.090,
        "escalate": 0.130,
    }

    action_cost = {
        "observe": 0.005,
        "monitor": 0.025,
        "investigate": 0.055,
        "contain": 0.095,
        "escalate": 0.145,
    }

    resilience = 1.0
    workload = 0.0
    cumulative_risk = 0.0
    rows = []
    n_events = max(len(df), 1)

    for t, row in df.iterrows():
        risk = float(row["adaptive_free_energy"])
        action = row["action"]

        if dyn["risk_volume_scaling"]:
            volume_factor = float(np.clip(np.sqrt(150 / n_events), 0.10, 1.00))
        else:
            volume_factor = 1.0

        effective_risk = risk * volume_factor

        cumulative_risk = dyn["risk_decay"] * cumulative_risk + effective_risk
        workload = dyn["workload_decay"] * workload + action_cost[action]

        passive_recovery = dyn["passive_recovery"] * (1.0 - resilience)
        risk_drain = dyn["risk_drain"] * effective_risk
        mitigation_gain = dyn["mitigation_gain"] * mitigation[action]
        workload_penalty = dyn["workload_penalty"] * np.tanh(workload)

        resilience = np.clip(
            resilience + passive_recovery - risk_drain + mitigation_gain - workload_penalty,
            0,
            1
        )

        rows.append({
            "dataset": dataset_name,
            "step": t,
            "risk": risk,
            "effective_risk": effective_risk,
            "action": action,
            "cumulative_risk": cumulative_risk,
            "workload": workload,
            "passive_recovery": passive_recovery,
            "risk_drain": risk_drain,
            "mitigation_gain": mitigation_gain,
            "workload_penalty": workload_penalty,
            "resilience": resilience,
        })

    return pd.DataFrame(rows)

twin_df = pd.concat([
    simulate_digital_twin(decision_df, "cyber"),
    simulate_digital_twin(decision_df, "video"),
], ignore_index=True)

save_table(twin_df, "table_digital_twin_state.csv")

twin_summary = twin_df.groupby("dataset").agg(
    mean_risk=("risk", "mean"),
    mean_effective_risk=("effective_risk", "mean"),
    max_cumulative_risk=("cumulative_risk", "max"),
    mean_workload=("workload", "mean"),
    final_resilience=("resilience", "last"),
    min_resilience=("resilience", "min"),
    mean_recovery=("passive_recovery", "mean"),
    mean_mitigation=("mitigation_gain", "mean"),
).reset_index()

save_table(twin_summary, "table_digital_twin_summary.csv")
display(twin_summary)

In [ ]:
# ============================================================
# Cell 12 — Core performance figures
# ============================================================

plt.figure(figsize=(9, 5))
metrics_df.pivot(index="model", columns="dataset", values="f1").plot(kind="bar", ax=plt.gca())
plt.title("Model F1-score comparison")
plt.ylabel("F1-score")
plt.xlabel("Model")
plt.xticks(rotation=35, ha="right")
savefig("fig_01_model_f1_comparison.png")

plt.figure(figsize=(9, 5))
metrics_df.pivot(index="model", columns="dataset", values="roc_auc").plot(kind="bar", ax=plt.gca())
plt.title("Model ROC-AUC comparison")
plt.ylabel("ROC-AUC")
plt.xlabel("Model")
plt.xticks(rotation=35, ha="right")
savefig("fig_02_model_roc_auc_comparison.png")

plt.figure(figsize=(9, 5))
metrics_df.pivot(index="model", columns="dataset", values="balanced_accuracy").plot(kind="bar", ax=plt.gca())
plt.title("Balanced accuracy comparison")
plt.ylabel("Balanced accuracy")
plt.xlabel("Model")
plt.xticks(rotation=35, ha="right")
savefig("fig_03_balanced_accuracy_comparison.png")

In [ ]:
# ============================================================
# Cell 13 — Confusion matrices and classification reports
# ============================================================

classification_report_rows = []

for name, run in [("cyber", cyber_run), ("video", video_run)]:
    proba = run["best_proba"]

    if proba.shape[1] == 2:
        pred = (proba[:, 1] >= 0.5).astype(int)
    else:
        pred = np.argmax(proba, axis=1)

    cm = confusion_matrix(run["y_test"], pred)

    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(cm)
    disp.plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(f"Confusion matrix — {name}")
    savefig(f"fig_04_confusion_matrix_{name}.png")

    report_dict = classification_report(run["y_test"], pred, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).T.reset_index().rename(columns={"index": "class_or_average"})
    report_df["dataset"] = name
    classification_report_rows.append(report_df)

classification_reports_df = pd.concat(classification_report_rows, ignore_index=True)
save_table(classification_reports_df, "table_classification_reports.csv")
display(classification_reports_df.head(12))

In [ ]:
# ============================================================
# Cell 14 — Calibration and reliability figures
# ============================================================

def calibration_data(run):
    y = np.asarray(run["y_test"])
    proba = np.asarray(run["best_proba"])

    if proba.shape[1] == 2:
        return y, proba[:, 1], "positive probability"

    pred = np.argmax(proba, axis=1)
    correctness = (pred == y).astype(int)
    confidence = np.max(proba, axis=1)
    return correctness, confidence, "confidence"

calibration_rows = []

for name, run in [("cyber", cyber_run), ("video", video_run)]:
    y_calib, p_calib, xlabel = calibration_data(run)

    frac_pos, mean_pred = calibration_curve(y_calib, p_calib, n_bins=10, strategy="uniform")

    cal_df = pd.DataFrame({
        "dataset": name,
        "mean_predicted": mean_pred,
        "observed_fraction": frac_pos,
    })
    calibration_rows.append(cal_df)

    plt.figure(figsize=(6, 5))
    plt.plot(mean_pred, frac_pos, marker="o", label=name)
    plt.plot([0, 1], [0, 1], linestyle="--", label="perfect calibration")
    plt.title(f"Calibration / reliability curve — {name}")
    plt.xlabel(f"Mean predicted {xlabel}")
    plt.ylabel("Observed correctness / positive rate")
    plt.legend()
    savefig(f"fig_05_calibration_{name}.png")

calibration_df = pd.concat(calibration_rows, ignore_index=True)
save_table(calibration_df, "table_calibration_bins.csv")
display(calibration_df.head())

In [ ]:
# ============================================================
# Cell 15 — Free-energy, action, and decision figures
# ============================================================

plt.figure(figsize=(8, 5))
for ds in ["cyber", "video"]:
    subset = decision_df[decision_df["dataset"] == ds]
    plt.hist(subset["adaptive_free_energy"], bins=30, alpha=0.5, label=ds)
plt.title("Adaptive free-energy distribution")
plt.xlabel("Adaptive free energy")
plt.ylabel("Frequency")
plt.legend()
savefig("fig_06_adaptive_free_energy_distribution.png")

plt.figure(figsize=(9, 5))
action_pivot = action_counts.pivot(index="action", columns="dataset", values="count").fillna(0)
action_pivot.plot(kind="bar", ax=plt.gca())
plt.title("Operational action distribution")
plt.ylabel("Count")
plt.xlabel("Action")
plt.xticks(rotation=35, ha="right")
savefig("fig_07_action_distribution.png")

plt.figure(figsize=(7, 5))
for ds in ["cyber", "video"]:
    subset = decision_df[decision_df["dataset"] == ds]
    plt.scatter(subset["uncertainty"], subset["adaptive_free_energy"], alpha=0.35, label=ds)
plt.title("Uncertainty versus adaptive free energy")
plt.xlabel("Uncertainty")
plt.ylabel("Adaptive free energy")
plt.legend()
savefig("fig_08_uncertainty_vs_free_energy.png")

plt.figure(figsize=(7, 5))
for ds in ["cyber", "video"]:
    subset = decision_df[decision_df["dataset"] == ds]
    plt.scatter(subset["model_risk"], subset["adaptive_free_energy"], alpha=0.35, label=ds)
plt.title("Model risk versus adaptive free energy")
plt.xlabel("Model risk")
plt.ylabel("Adaptive free energy")
plt.legend()
savefig("fig_09_model_risk_vs_free_energy.png")

In [ ]:
# ============================================================
# Cell 16 — Digital twin figures
# ============================================================

plt.figure(figsize=(9, 5))
for ds in ["cyber", "video"]:
    subset = twin_df[twin_df["dataset"] == ds]
    plt.plot(subset["step"], subset["resilience"], label=ds)
plt.title("Digital twin resilience trajectories")
plt.xlabel("Simulation step")
plt.ylabel("Resilience")
plt.legend()
savefig("fig_10_digital_twin_resilience.png")

plt.figure(figsize=(9, 5))
for ds in ["cyber", "video"]:
    subset = twin_df[twin_df["dataset"] == ds]
    plt.plot(subset["step"], subset["effective_risk"], label=ds)
plt.title("Volume-aware effective risk trajectory")
plt.xlabel("Simulation step")
plt.ylabel("Effective risk")
plt.legend()
savefig("fig_11_effective_risk_trajectory.png")

plt.figure(figsize=(9, 5))
for ds in ["cyber", "video"]:
    subset = twin_df[twin_df["dataset"] == ds]
    plt.plot(subset["step"], subset["workload"], label=ds)
plt.title("Digital twin workload trajectories")
plt.xlabel("Simulation step")
plt.ylabel("Workload")
plt.legend()
savefig("fig_12_workload_trajectory.png")

plt.figure(figsize=(7, 5))
for ds in ["cyber", "video"]:
    subset = twin_df[twin_df["dataset"] == ds]
    plt.scatter(subset["effective_risk"], subset["resilience"], alpha=0.35, label=ds)
plt.title("Effective risk versus resilience")
plt.xlabel("Effective risk")
plt.ylabel("Resilience")
plt.legend()
savefig("fig_13_effective_risk_vs_resilience.png")

In [ ]:
# ============================================================
# Cell 17 — Explainability: permutation importance
# ============================================================

def get_perm_scoring(run):
    if run["best_proba"].shape[1] == 2:
        return "f1"
    return make_scorer(f1_score, average="weighted", zero_division=0)

importance_tables = []

for name, run in [("cyber", cyber_run), ("video", video_run)]:
    print("Computing permutation importance for:", name)

    result = permutation_importance(
        run["best_model"],
        run["X_test"],
        run["y_test"],
        n_repeats=6,
        random_state=SEED,
        scoring=get_perm_scoring(run),
        n_jobs=-1,
    )

    imp_df = pd.DataFrame({
        "dataset": name,
        "feature": run["X_test"].columns,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std,
    }).sort_values("importance_mean", ascending=False).reset_index(drop=True)

    importance_tables.append(imp_df)

    save_table(imp_df, f"table_permutation_importance_{name}.csv")

    top = imp_df.head(CONFIG["max_permutation_features"])

    plt.figure(figsize=(8, 5))
    plt.barh(top["feature"][::-1], top["importance_mean"][::-1])
    plt.title(f"Top permutation importances — {name}")
    plt.xlabel("Mean importance")
    savefig(f"fig_14_permutation_importance_{name}.png")

importance_df = pd.concat(importance_tables, ignore_index=True)
save_table(importance_df, "table_permutation_importance_all.csv")
display(importance_df.head(20))

In [ ]:
# ============================================================
# Cell 18 — Robustness under probability perturbation
# ============================================================

def robustness_curve(run, dataset_name):
    y = np.asarray(run["y_test"])
    base_p = np.asarray(run["best_proba"])
    n_classes = base_p.shape[1]

    rng = np.random.default_rng(SEED)
    rows = []

    for noise in np.linspace(0, 0.25, 8):
        scores = []

        for _ in range(20):
            noisy = base_p + rng.normal(0, noise, size=base_p.shape)
            noisy = np.clip(noisy, 1e-8, None)
            noisy = noisy / noisy.sum(axis=1, keepdims=True)

            if n_classes == 2:
                pred = (noisy[:, 1] >= 0.5).astype(int)
                score = f1_score(y, pred, zero_division=0)
            else:
                pred = np.argmax(noisy, axis=1)
                score = f1_score(y, pred, average="weighted", zero_division=0)

            scores.append(score)

        rows.append({
            "dataset": dataset_name,
            "noise": noise,
            "f1_mean": float(np.mean(scores)),
            "f1_std": float(np.std(scores)),
        })

    return pd.DataFrame(rows)

robust_df = pd.concat([
    robustness_curve(cyber_run, "cyber"),
    robustness_curve(video_run, "video"),
], ignore_index=True)

save_table(robust_df, "table_probability_robustness.csv")

plt.figure(figsize=(8, 5))
for ds in ["cyber", "video"]:
    subset = robust_df[robust_df["dataset"] == ds]
    plt.errorbar(subset["noise"], subset["f1_mean"], yerr=subset["f1_std"], marker="o", label=ds)
plt.title("Robustness under probability perturbation")
plt.xlabel("Noise standard deviation")
plt.ylabel("F1-score")
plt.legend()
savefig("fig_15_probability_robustness.png")

display(robust_df)

In [ ]:
# ============================================================
# Cell 19 — Fast graph-based operational interpretation
# ============================================================

graph_summary = {}

if HAS_NETWORKX:
    G = nx.Graph()

    sample_df = decision_df.sample(min(120, len(decision_df)), random_state=SEED)

    for idx, row in sample_df.iterrows():
        event_node = f"{row['dataset']}_event_{idx}"
        action_node = f"action_{row['action']}"
        energy_node = f"energy_{int(row['adaptive_free_energy'] * 10)}"
        uncertainty_node = f"uncertainty_{int(row['uncertainty'] * 10)}"

        G.add_edge(event_node, action_node)
        G.add_edge(event_node, energy_node)
        G.add_edge(event_node, uncertainty_node)

    degree = dict(G.degree())
    central_nodes = sorted(degree.items(), key=lambda x: x[1], reverse=True)[:10]

    graph_summary = {
        "num_nodes": G.number_of_nodes(),
        "num_edges": G.number_of_edges(),
        "sample_size": len(sample_df),
        "top_degree_nodes": central_nodes,
    }

    save_json(graph_summary, "graph_summary.json")

    plt.figure(figsize=(9, 7))

    try:
        pos = nx.kamada_kawai_layout(G)
    except Exception:
        pos = nx.random_layout(G, seed=SEED)

    nx.draw_networkx_nodes(G, pos, node_size=[25 + 6 * degree[n] for n in G.nodes()], alpha=0.75)
    nx.draw_networkx_edges(G, pos, alpha=0.15, width=0.7)

    label_nodes = {n: n for n, _ in central_nodes[:6]}
    nx.draw_networkx_labels(G, pos, labels=label_nodes, font_size=8)

    plt.title("Incident-action-free-energy graph")
    plt.axis("off")
    savefig("fig_16_incident_action_free_energy_graph.png")

else:
    graph_summary = {"networkx_available": False}
    save_json(graph_summary, "graph_summary.json")

graph_summary

In [ ]:
# ============================================================
# Cell 20 — Save models
# ============================================================

model_paths = {}

for name, run in [("cyber", cyber_run), ("video", video_run)]:
    path = MODEL_DIR / f"best_{name}_model.pkl"
    with open(path, "wb") as f:
        pickle.dump(run["best_model"], f)
    model_paths[name] = str(path)
    print("Saved model:", path)

save_json(model_paths, "model_paths.json")

In [ ]:
# ============================================================
# Cell 21 — Final outputs summary
# ============================================================

final_summary = {
    "framework_name": CONFIG["framework_name"],
    "framework_abbreviation": CONFIG["framework_abbreviation"],
    "base_dir": str(BASE_DIR),
    "cyber_path": str(cyber_path),
    "video_path": str(video_path),
    "best_cyber_model": cyber_run["best_name"],
    "best_video_model": video_run["best_name"],
    "cyber_n_classes": cyber_run["n_classes"],
    "video_n_classes": video_run["n_classes"],
    "figures_dir": str(FIG_DIR),
    "tables_dir": str(TABLE_DIR),
    "models_dir": str(MODEL_DIR),
    "outputs_dir": str(OUTPUT_DIR),
    "num_figures": len(list(FIG_DIR.glob("*.png"))),
    "num_tables": len(list(TABLE_DIR.glob("*.csv"))),
    "best_model_summary": best_model_summary.to_dict(orient="records"),
    "decision_summary": decision_summary.to_dict(orient="records"),
    "digital_twin_summary": twin_summary.to_dict(orient="records"),
    "graph_summary": graph_summary,
}

save_json(final_summary, "run_summary.json")

summary_lines = []
summary_lines.append("DTDIF — Digital Twin Decision Intelligence Framework")
summary_lines.append("Final complete Colab + Google Drive notebook")
summary_lines.append("=" * 72)
summary_lines.append("")
summary_lines.append(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"BASE_DIR: {BASE_DIR}")
summary_lines.append("")
summary_lines.append("Generated folders:")
summary_lines.append(f" - Figures: {FIG_DIR}")
summary_lines.append(f" - Tables: {TABLE_DIR}")
summary_lines.append(f" - Models: {MODEL_DIR}")
summary_lines.append(f" - Outputs: {OUTPUT_DIR}")
summary_lines.append("")
summary_lines.append("Best models:")
summary_lines.append(f" - Cyber: {cyber_run['best_name']} | classes: {cyber_run['n_classes']}")
summary_lines.append(f" - Video: {video_run['best_name']} | classes: {video_run['n_classes']}")
summary_lines.append("")
summary_lines.append("Best model summary:")
summary_lines.append(best_model_summary.to_string(index=False))
summary_lines.append("")
summary_lines.append("Decision summary:")
summary_lines.append(decision_summary.to_string(index=False))
summary_lines.append("")
summary_lines.append("Digital twin summary:")
summary_lines.append(twin_summary.to_string(index=False))
summary_lines.append("")
summary_lines.append("Action counts:")
summary_lines.append(action_counts.to_string(index=False))
summary_lines.append("")
summary_lines.append("Generated figures:")
for f in sorted(FIG_DIR.glob("*.png")):
    summary_lines.append(f" - {f.name}")
summary_lines.append("")
summary_lines.append("Generated tables:")
for f in sorted(TABLE_DIR.glob("*.csv")):
    summary_lines.append(f" - {f.name}")
summary_lines.append("")
summary_lines.append("Generated models:")
for f in sorted(MODEL_DIR.glob("*.pkl")):
    summary_lines.append(f" - {f.name}")
summary_lines.append("")
summary_lines.append("Run summary JSON:")
summary_lines.append(str(OUTPUT_DIR / "run_summary.json"))

SUMMARY_PATH.write_text("\n".join(summary_lines), encoding="utf-8")

print("\n" + "=" * 72)
print("FINAL OUTPUTS CREATED")
print("=" * 72)
print("Figures:", len(list(FIG_DIR.glob('*.png'))))
print("Tables:", len(list(TABLE_DIR.glob('*.csv'))))
print("Models:", len(list(MODEL_DIR.glob('*.pkl'))))
print("Outputs summary:", SUMMARY_PATH)
print("=" * 72)

## Expected final outputs

After running all cells, the notebook will produce:

### Figures
At least 16 PNG figures, including model comparison, confusion matrices, calibration curves, free-energy distributions, action distributions, twin resilience, robustness, feature importance, and graph interpretation.

### Tables
Multiple CSV tables, including dataset overview, target distribution, model performance, best models, CV stability, decision outputs, action counts, digital-twin state, classification reports, calibration bins, permutation importance, robustness, and summary tables.

### Summary
The file `outputs_summary.txt` is saved in:

`/content/drive/MyDrive/Outputs/DTDIF_Final/outputs/outputs_summary.txt`